In [ ]:
# 1. Load the saved results and models from artifacts
import json
import os
from pathlib import Path

import gradio as gr
import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from sklearn.metrics import confusion_matrix

ART = Path("artifacts")
summary = json.loads((ART / "metrics.json").read_text())
df = pd.read_csv(ART / "predictions.csv", dtype={"venue_id": str})

MODELS = ["scikit-learn", "PyTorch by hand", "PyTorch standard"]
SPLITS = {"Test (2024 venues)": "test", "Validation (2023 venues)": "validation"}
COSTS = summary["costs"]
TARGET = "did_not_survive"

In [ ]:
# 2. Re-score every venue with the three saved models and check they match training
def build_model(n_features):
    return torch.nn.Sequential(torch.nn.Linear(n_features, 1), torch.nn.Sigmoid())


preprocess = joblib.load(ART / "preprocess.joblib")
sk_model = joblib.load(ART / "model_sklearn.joblib")
by_hand = torch.load(ART / "model_pytorch_by_hand.pt")
standard = build_model(len(summary["features"]["encoded"]))
standard.load_state_dict(torch.load(ART / "model_pytorch_standard.pt"))
standard.eval()

X = torch.from_numpy(preprocess.transform(df).astype(np.float32))
with torch.no_grad():
    rescored = {
        "scikit-learn": sk_model.predict_proba(X.numpy())[:, 1],
        "PyTorch by hand": torch.sigmoid(X @ by_hand["w"] + by_hand["b"]).numpy(),
        "PyTorch standard": standard(X).squeeze(-1).numpy(),
    }
for name, p in rescored.items():
    assert np.allclose(p, df[f"p::{name}"], atol=1e-4), f"{name} does not match training"
print("loaded the 3 saved models; their predictions match training")

In [ ]:
 # 3. Chart helpers: one colour and line style per model
STYLE = {
    "scikit-learn": dict(color="#2a78d6", dash="solid", symbol="circle"),
    "PyTorch by hand": dict(color="#eb6834", dash="dash", symbol="diamond"),
    "PyTorch standard": dict(color="#1baf7a", dash="dot", symbol="square"),
    "Baseline: district average": dict(color="#8a8984", dash="dashdot", symbol="x"),
}


def style(fig, title, x_title, y_title, height=420):
    fig.update_layout(title=title, template="plotly_white", height=height,
                      margin=dict(l=60, r=20, t=50, b=100),
                      legend=dict(orientation="h", y=-0.25, x=0))
    fig.update_xaxes(title=x_title)
    fig.update_yaxes(title=y_title)
    return fig

In [ ]:
# 4. Model comparison: predicted vs actual failure rate (venues grouped into 10 risk bins)
def predicted_vs_actual(split_label):
    d = df[df.split == SPLITS[split_label]]
    fig = go.Figure()
    for name in MODELS + ["Baseline: district average"]:
        bins = pd.qcut(d[f"p::{name}"].rank(method="first"), 10, labels=False)
        g = d.groupby(bins).agg(predicted=(f"p::{name}", "mean"), actual=(TARGET, "mean"))
        s = STYLE[name]
        fig.add_trace(go.Scatter(x=g.predicted, y=g.actual, mode="lines+markers", name=name,
                                 line=dict(color=s["color"], dash=s["dash"]),
                                 marker=dict(symbol=s["symbol"], size=8)))
    top = 0.2
    fig.add_trace(go.Scatter(x=[0, top], y=[0, top], mode="lines", name="perfect prediction",
                             line=dict(color="grey", dash="dot", width=1)))
    fig = style(fig, "Predicted vs actual failure rate", "average predicted risk", "actual failure rate")
    fig.update_xaxes(tickformat=".0%", range=[0, top])
    fig.update_yaxes(tickformat=".0%", range=[0, top])
    return fig


def results_table():
    r = pd.DataFrame(summary["results"])
    return r[["split", "model", "roc_auc", "pr_auc", "log_loss", "avg_predicted", "actual_rate"]].round(4)


def agreement_table():
    return pd.DataFrame(summary["agreement"]).round(6)


def risk_groups(split_label):
    # put every venue in a low / medium / high risk group and check how many really failed
    d = df[df.split == SPLITS[split_label]]
    group = pd.cut(d["p::scikit-learn"], [0, 0.05, COSTS["threshold"], 1],
                   labels=["low risk (< 5%)", f"medium risk (5% to {COSTS['threshold']:.0%})",
                           f"high risk (>= {COSTS['threshold']:.0%})"])
    g = d.groupby(group, observed=True)[TARGET].agg(venues="size", did_not_survive="sum", rate="mean")
    g["actual failure rate"] = (g.pop("rate") * 100).round(1).astype(str) + "%"
    return g.reset_index(names="risk group")

In [ ]:
# 5. Distributions: the target by year, and the failure rate for each feature value
FEATURES = ["district", "venue_type", "setting", "terrace", "not_street_level", "mixed_use",
            "competitors_250m"]


def target_chart():
    t = df.groupby(["cohort", "outcome"]).size().unstack(fill_value=0)
    share = t.div(t.sum(axis=1), axis=0)
    fig = go.Figure()
    for outcome in ["new_business", "changed_activity", "closed"]:
        fig.add_trace(go.Bar(x=[f"{c} → {c + 1}" for c in share.index], y=share[outcome],
                             name=outcome.replace("_", " ")))
    fig = style(fig, "Share of venues that did not survive the year", "year", "share of venues")
    fig.update_layout(barmode="stack")
    fig.update_yaxes(tickformat=".0%")
    return fig


def feature_chart(feature):
    d = df[df.split == "train"].copy()
    if feature == "competitors_250m":                       # group the counts so they can be compared
        d[feature] = pd.cut(d[feature], [-1, 5, 15, 30, 60, 1000],
                            labels=["0-5", "6-15", "16-30", "31-60", "60+"])
    g = d.groupby(feature, observed=True)[TARGET].agg(["mean", "size"]).sort_values("mean")
    fig = go.Figure(go.Bar(x=g["mean"], y=g.index.astype(str), orientation="h",
                           customdata=g["size"], hovertemplate="%{y}: %{x:.1%} (%{customdata} venues)"))
    fig.add_vline(x=d[TARGET].mean(), line_dash="dot", line_color="grey")
    fig = style(fig, f"Failure rate by {feature} (training data; dotted line = average)",
                "failure rate", "", height=max(380, 25 * len(g) + 120))
    fig.update_xaxes(tickformat=".0%")
    return fig

In [1]:
# 6. Threshold and business cost: confusion matrix and cost change with the slider
def threshold_view(split_label, model, threshold, cost_missed, cost_false_alarm):
    d = df[df.split == SPLITS[split_label]]
    y, p = d[TARGET].to_numpy(), d[f"p::{model}"].to_numpy()
    tn, fp, fn, tp = confusion_matrix(y, p >= threshold, labels=[0, 1]).ravel()
    total = fn * cost_missed + fp * cost_false_alarm
    nobody = y.sum() * cost_missed
    everyone = (1 - y).sum() * cost_false_alarm

    matrix = go.Figure(go.Heatmap(
        z=[[tn, fp], [fn, tp]], x=["predicted: survives", "predicted: at risk"],
        y=["actual: survived", "actual: did not survive"], colorscale="Blues", showscale=False,
        text=[[f"{tn:,} correct", f"{fp:,} false alarms"], [f"{fn:,} missed", f"{tp:,} caught"]],
        texttemplate="%{text}"))
    matrix = style(matrix, f"Confusion matrix at threshold {threshold:.3f}", "", "", height=360)
    matrix.update_yaxes(autorange="reversed")

    text = f"""
### Total cost: €{total:,.0f}
Deposit asked from {fp + tp:,} of {len(y):,} venues · {tp:,} of {int(y.sum()):,} failing venues covered · {fp:,} healthy venues asked

| Policy | Cost |
|---|---|
| **{model}, threshold {threshold:.3f}** | **€{total:,.0f}** |
| Deposit from nobody | €{nobody:,.0f} |
| Deposit from everyone | €{everyone:,.0f} |
"""
    return matrix, text

In [ ]:
# 7. Build the dashboard and launch it
with gr.Blocks(title="Madrid venue survival") as demo:
    gr.Markdown("# Which Madrid bars & restaurants won't survive the year?\n"
                "Logistic regression trained three ways. All models are loaded from `artifacts/`.")
    split_names = list(SPLITS)

    with gr.Tab("Risk groups"):
        gr.Markdown("Every venue gets a predicted risk of not surviving the next year. "
                    "Grouping them shows whether the high-risk group really failed more often.")
        split_r = gr.Radio(split_names, value=split_names[0], label="Data")
        table_r = gr.Dataframe(risk_groups(split_names[0]))
        split_r.change(risk_groups, split_r, table_r)

    with gr.Tab("Model comparison"):
        split_a = gr.Radio(split_names, value=split_names[0], label="Data")
        plot_a = gr.Plot(predicted_vs_actual(split_names[0]))
        split_a.change(predicted_vs_actual, split_a, plot_a)
        gr.Dataframe(results_table(), label="Scores")
        gr.Dataframe(agreement_table(), label="Do the three versions agree?")

    with gr.Tab("Distributions"):
        gr.Plot(target_chart())
        feature = gr.Dropdown(FEATURES, value="district", label="Feature")
        plot_b = gr.Plot(feature_chart("district"))
        feature.change(feature_chart, feature, plot_b)

    with gr.Tab("Threshold & cost"):
        gr.Markdown("Before installing free equipment, the brewery **asks for a deposit** from every "
                    "venue whose predicted risk ≥ threshold.")
        with gr.Row():
            split_c = gr.Radio(split_names, value=split_names[0], label="Data")
            model_c = gr.Dropdown(MODELS, value="scikit-learn", label="Model")
        threshold = gr.Slider(0.01, 0.3, value=COSTS["threshold"], step=0.005, label="Threshold")
        with gr.Row():
            cost_missed = gr.Number(value=COSTS["missed"], label="Cost of a missed failure: equipment lost (€)")
            cost_false = gr.Number(value=COSTS["false_alarm"], label="Cost of a false alarm: healthy venue lost (€)")
        cost_text = gr.Markdown()
        matrix_plot = gr.Plot()
        inputs = [split_c, model_c, threshold, cost_missed, cost_false]
        for control in inputs:
            control.change(threshold_view, inputs, [matrix_plot, cost_text])
        demo.load(threshold_view, inputs, [matrix_plot, cost_text])

demo.launch(server_name=os.environ.get("GRADIO_SERVER_NAME", "127.0.0.1"), server_port=7860)